# 직접 해보기 09 · 341개 목록에서 표본을 직접 고르기

Kocevski 외(2025)가 공개한 작은 붉은 점 후보 341개 목록(저자 저장소 커밋 b47c187)에서 밝기 조건을 걸어 표본을 고른다. 같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
rows = A.load()
print(A.sha8(), len(rows), '행, 분광', sum(r['flag'] for r in rows), '개')

## 1. 분모부터 확인한다
전체 341개, 분광 39개가 이후 모든 비율의 분모다.

In [ ]:
from collections import Counter
print(Counter(r['survey'] for r in rows))
print('분광', sum(r['flag'] for r in rows), '/ 전체', len(rows))

## 2. 밝기 조건을 건다
`mag444 < 한계`. 같다는 포함하지 않는다.

In [ ]:
for lim in (26, 26.5, 27):
    s = A.summarize(rows, lim)
    print(f"mag444<{lim}: 남음 {s['kept']} (분광 {s['kept_spec']}), 빠짐 {s['dropped']}")

## 3. 분포를 비교한다
matplotlib이 없으면 `pip install matplotlib`.

In [ ]:
import matplotlib.pyplot as plt
s = A.summarize(rows, 26.5)
b = list(A.BINS)
x = [v + .5 for v in b]
plt.figure(figsize=(8, 4))
plt.bar(x, [s['per_bin'][v]['all'] for v in b], width=.8, color='#B9B3A6', label='전체 341')
plt.bar(x, [s['per_bin'][v]['kept'] for v in b], width=.8, color='#B8475E', label='mag444 < 26.5')
plt.axvline(s['median_kept'], color='#B8475E'); plt.axvline(s['median_dropped'], color='#D9733D')
plt.xlabel('redshift z'); plt.ylabel('count'); plt.legend(); plt.show()
print('중앙값 전체/통과/빠짐', s['median_all'], s['median_kept'], s['median_dropped'])

## 4. 구간마다 남은 비율
먼 구간일수록 더 많이 빠진다. 이것이 ‘조건이 분포의 모양을 함께 만든다’는 뜻이다.

In [ ]:
for v in b:
    p = s['per_bin'][v]
    print(f"z {v}-{v+1}: {p['kept']}/{p['all']} = {p['pct']}%")

## 5. 기록을 남긴다
조건, 남은 수, 분모, 파일 지문을 적는다. 학술지 PDF의 Table 3과는 zflag 표기가 다르다는 점도 함께 적자.

In [ ]:
print(f"조건 mag444<26.5 · 남음 {s['kept']} · 분모 {s['denominator']} · catalog.csv sha256 {A.sha8()}")